In [1]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import NoiseSensitivity

client = AsyncOpenAI()
llm = llm_factory("gpt-5-mini", client=client)

scorer = NoiseSensitivity(llm=llm)

/Users/satvik/Desktop/Project/personal/RAGAS/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Example 1: Response is mostly correct but 'contributes to financial stability'
# is a claim drawn from the noisy context rather than the core reference
result = await scorer.ascore(
    user_input="What is the Life Insurance Corporation of India (LIC) known for?",
    response="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, known for its vast portfolio of investments. LIC contributes to the financial stability of the country.",
    reference="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, established in 1956 through the nationalization of the insurance industry. It is known for managing a large portfolio of investments.",
    retrieved_contexts=[
        "The Life Insurance Corporation of India (LIC) was established in 1956 following the nationalization of the insurance industry in India.",
        "LIC is the largest insurance company in India, with a vast network of policyholders and huge investments.",
        "As the largest institutional investor in India, LIC manages substantial funds, contributing to the financial stability of the country.",
        "The Indian economy is one of the fastest-growing major economies in the world, thanks to sectors like finance, technology, and manufacturing."
    ]
)
print(f"Noise Sensitivity Score: {result.value}")

Noise Sensitivity Score: 0.3333333333333333


In [3]:
# Example 2 : Response stays grounded in the reference despite noisy contexts
# The Louvre and French culture contexts are completely ignored by the response
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris, France.",
    reference="The Eiffel Tower is located in Paris, France.",
    retrieved_contexts=[
        "The Eiffel Tower is a landmark located in Paris, France.",
        "Paris is also home to the Louvre Museum, which contains thousands of artworks including the Mona Lisa.",
        "France is known for its cuisine, wine, and fashion industry."
    ]
)
print(f"Noise Sensitivity Score: {result.value}")

Noise Sensitivity Score: 0.0
